In [30]:
import pandas as pd
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)
from sklearn.model_selection import train_test_split



dataset = pd.read_csv('train-2.csv')
print(dataset.shape)
print(dataset.isnull().sum())
dataset.head()

(891, 12)
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [31]:
dict_for_coding = {'male' : 1, 'female' : 0}

dataset['Sex_coding'] = [dict_for_coding[i] for i in dataset['Sex']]
features_for_predict = ['Pclass', 'Sex_coding','Age','Fare',]
X,y = dataset[features_for_predict], dataset['Survived']




pipeline = make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), LogisticRegression())
cross_val = cross_val_score(pipeline,X,y,cv=5,scoring='accuracy')
print(cross_val.mean())
print(cross_val.std())

0.7856317870817902
0.0055734018132204645


In [32]:
new_features_for_predict = ['Pclass', 'Sex_coding','Age','Fare', 'SibSp','Parch']
X,y = dataset[new_features_for_predict],dataset['Survived']
cross_val = cross_val_score(pipeline,X,y,cv=5,scoring='accuracy')
print(cross_val.mean())
print(cross_val.std())

0.7856317870817903
0.022319846461439415


После добавление SibSp и Parch среднее стало немного меньше (0.7856 раньше, 0.7865 стало)
std так же уменьшлися (было 0.005 стало 0.0223)


In [33]:
dataset['family_size'] = dataset['SibSp'] + dataset['Parch'] + 1

new_features_for_predict1 = ['Pclass', 'Sex_coding','Age','Fare', 'family_size']
X, y = dataset[new_features_for_predict1], dataset['Survived']

cross_val = cross_val_score(pipeline,X,y,cv=5,scoring='accuracy')
print(cross_val.mean())
print(cross_val.std())

0.7912497646098802
0.01749644020308927


Нет Family_size не нужен с SibSp и Parch ведь он включает в себя это

In [34]:
most_frequent = dataset['Embarked'].mode()[0]
dataset['Embarked'] = dataset['Embarked'].fillna(most_frequent)
dataset = pd.get_dummies(dataset, columns=['Embarked'], dtype=int)


new_features_for_predict1 = ['Pclass', 'Sex_coding','Age','Fare', 'family_size','Embarked_C','Embarked_Q','Embarked_S']
X, y = dataset[new_features_for_predict1], dataset['Survived']

cross_val = cross_val_score(pipeline,X,y,cv=5,scoring='accuracy')
print(cross_val.mean())
print(cross_val.std())



0.7879040863724813
0.016062522600672032


In [35]:
dataset['Title'] = dataset['Name'].str.split(', ').str[1].str.split('.').str[0]
dataset['Title'] = dataset['Title'].replace({'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs'})
main_titles = ['Mr', 'Miss', 'Mrs', 'Master']
dataset['Title'] = dataset['Title'].where(dataset['Title'].isin(main_titles), 'Rare')
dataset = pd.get_dummies(dataset, columns=['Title'], dtype=int)
new_features_for_predict1 = ['Pclass', 'Sex_coding','Age','Fare', 'family_size','Embarked_C','Embarked_Q','Embarked_S','Title_Master',	'Title_Miss',	'Title_Mr',	'Title_Mrs',	'Title_Rare']
X, y = dataset[new_features_for_predict1], dataset['Survived']

pipeline = make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), LogisticRegression(C=1))
cross_val = cross_val_score(pipeline,X,y,cv=5,scoring='accuracy')
print(cross_val.mean())
print(cross_val.std())


0.8271608813006088
0.025959034730134334


По сравнению со всеми прошлыми резульатами (см.выше) этот результат явно лучший и обгоняет даже прошлый лучший результат(0.7912)

Оставляем С = 1

In [36]:
# 1. Обучаем пайплайн на всех 891 строках
print(X.shape)   # (891, 13)
pipeline.fit(X, y)

# 2. Преобразования test.csv (те же, что у train)
test = pd.read_csv('test-2.csv')

test['Sex_coding'] = test['Sex'].map({'male': 1, 'female': 0})
test['family_size'] = test['SibSp'] + test['Parch'] + 1
test['Embarked'] = test['Embarked'].fillna('S')

test['Title'] = test['Name'].str.split(', ').str[1].str.split('.').str[0]
test['Title'] = test['Title'].replace({'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs'})
test['Title'] = test['Title'].where(test['Title'].isin(main_titles), 'Rare')

test = pd.get_dummies(test, columns=['Embarked', 'Title'], dtype=int)

# 3. Те же 13 признаков в том же порядке
X_test_final = test.reindex(columns=new_features_for_predict1, fill_value=0)
print(X_test_final.shape)         # (418, 13)
print(X_test_final.isna().sum())  # пропуски только в Age и Fare

# 4. Предсказание и файл
test['Survived'] = pipeline.predict(X_test_final)
test[['PassengerId', 'Survived']].to_csv('submission.csv', index=False)

# 5. Проверка
submission = pd.read_csv('submission.csv')
print(submission.shape)
print(submission['Survived'].value_counts())
print(submission['PassengerId'].min(), submission['PassengerId'].max())

(891, 13)
(418, 13)
Pclass           0
Sex_coding       0
Age             86
Fare             1
family_size      0
Embarked_C       0
Embarked_Q       0
Embarked_S       0
Title_Master     0
Title_Miss       0
Title_Mr         0
Title_Mrs        0
Title_Rare       0
dtype: int64
(418, 2)
Survived
0    251
1    167
Name: count, dtype: int64
892 1309
